In [ ]:
# -*- coding: utf-8 -*-
# ===============================
# ANTI-CRASH ENVIRONMENT & MEMORY MANAGEMENT
# ===============================
import os
import gc 
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' 
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True' 

import cv2, numpy as np, warnings
import tensorflow as tf
import librosa
import matplotlib.pyplot as plt
from moviepy import VideoFileClip
from tensorflow.keras import layers, models, regularizers, backend as K
from tensorflow.keras.applications import EfficientNetB0
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils import class_weight
import seaborn as sns

warnings.filterwarnings("ignore")

# ===============================
# CONFIG M6 (AUDIO + VISUAL)
# ===============================
BASE_DATASET = r"/var/www/html/video_clasification_fusion/Dataset" # Ganti sesuai path Anda
PROCESSED_DATA_FILE = "processed_data_m9_adaptive_cv.npz"
MODEL_FILE = "best_model_m6_audio_visual_cv.h5"

IMG_SIZE = (224, 224)
SPEC_SIZE = (128, 128)
SR = 22050
DURATION = 3
BATCH_SIZE = 8
EPOCHS = 20
KFOLD = 5  # 5-Fold CV
MAX_SAMPLES_PER_CLASS = 676 # Batas dataset 676 Kasar & 676 Non-Kasar

# ===============================
# HELPER FUNCTIONS
# ===============================
def find_category_dir(base_path, target_name):
    if not os.path.exists(base_path): return None
    for item in os.listdir(base_path):
        if item.lower() == target_name.lower() and os.path.isdir(os.path.join(base_path, item)):
            return os.path.join(base_path, item)
    return None

# ===============================
# FEATURE EXTRACTION (VISUAL + AUDIO)
# ===============================
def extract_frame(video_path):
    """Mengambil frame tengah dari video"""
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, total // 2)
    ret, frame = cap.read()
    cap.release()
    
    if not ret: 
        return np.zeros((*IMG_SIZE, 3), dtype=np.float32)
        
    frame = cv2.resize(frame, IMG_SIZE)
    return (frame / 255.0).astype(np.float32)

def extract_audio_spectrogram(video_path):
    """Mengekstrak audio dari video untuk diubah menjadi Mel-Spectrogram"""
    tmp = f"temp_proc_{os.getpid()}.wav"
    try:
        with VideoFileClip(video_path) as video:
            if video.audio is None: 
                video.close()
                return np.zeros((*SPEC_SIZE, 1), dtype=np.float32)
            video.audio.write_audiofile(tmp, logger=None)
            video.close() 
            
        y, sr = librosa.load(tmp, sr=SR)
        if len(y) < SR * DURATION: y = np.pad(y, (0, SR * DURATION - len(y)))
        y = y[:SR * DURATION]
        
        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
        mel = cv2.resize(librosa.power_to_db(mel), SPEC_SIZE)
        mel = np.expand_dims(mel, -1)
        mel = (mel - np.mean(mel)) / (np.std(mel) + 1e-9)
        
        if os.path.exists(tmp): os.remove(tmp)
        return mel.astype(np.float32)
    except Exception:
        if os.path.exists(tmp): os.remove(tmp)
        return np.zeros((*SPEC_SIZE, 1), dtype=np.float32)

# ===============================
# DATA LOADER
# ===============================
def load_dataset_bimodal():
    if os.path.exists(PROCESSED_DATA_FILE):
        d = np.load(PROCESSED_DATA_FILE)
        X_i_loaded, X_s_loaded, y_loaded = d['X_img'], d['X_spec'], d['y']
        if len(X_i_loaded) > 0:
            print(f"[INFO] Memuat {len(X_i_loaded)} data dari cache: {PROCESSED_DATA_FILE}")
            return X_i_loaded, X_s_loaded, y_loaded
        else:
            print(f"[WARNING] Cache kosong. Menghapus cache dan membaca ulang dataset...")
            os.remove(PROCESSED_DATA_FILE)
    
    path_to_use = BASE_DATASET if os.path.exists(BASE_DATASET) else os.path.join(os.getcwd(), "Dataset")
    print(f"Mengekstrak data AUDIO & VISUAL dari {path_to_use} (Mencegah Kernel Restart)...")
    
    total_target = MAX_SAMPLES_PER_CLASS * 2
    X_i = np.zeros((total_target, *IMG_SIZE, 3), dtype=np.float32)
    X_s = np.zeros((total_target, *SPEC_SIZE, 1), dtype=np.float32)
    Y = np.zeros((total_target,), dtype=np.int32)
    
    categories = {'nonkasar': 0, 'kasar': 1}
    idx = 0 
    
    for cat_name, label in categories.items():
        cat_path = find_category_dir(path_to_use, cat_name)
        if not cat_path: continue
        
        subfolders = [d for d in os.listdir(cat_path) if os.path.isdir(os.path.join(cat_path, d))]
        if not subfolders: subfolders = [""]
            
        quota_per_subfolder = MAX_SAMPLES_PER_CLASS // len(subfolders)
        remainder = MAX_SAMPLES_PER_CLASS % len(subfolders)
        class_count = 0 
        
        for sf_idx, sf in enumerate(subfolders):
            sf_path = os.path.join(cat_path, sf) if sf else cat_path
            current_quota = quota_per_subfolder + (1 if sf_idx < remainder else 0)
            sub_count = 0
            
            for root, dirs, files in os.walk(sf_path):
                if ".ipynb_checkpoints" in root: continue
                for f in files:
                    if class_count >= MAX_SAMPLES_PER_CLASS or sub_count >= current_quota: break 
                    if f.lower().endswith((".mp4", ".avi", ".mkv")):
                        full_path = os.path.join(root, f)
                        try:
                            img_a = extract_frame(full_path)
                            spec_a = extract_audio_spectrogram(full_path)
                            
                            X_i[idx] = img_a; X_s[idx] = spec_a; Y[idx] = label
                            sub_count += 1; class_count += 1; idx += 1
                            
                            folder_display = sf if sf else cat_name
                            print(f"[AUDIO+VISUAL | {cat_name.upper()} | {folder_display}] {sub_count}/{current_quota} LOADED: {f[:25]}")
                            
                            del img_a, spec_a
                            if class_count % 10 == 0: gc.collect()
                        except: continue
                if class_count >= MAX_SAMPLES_PER_CLASS or sub_count >= current_quota: break

    X_i, X_s, Y = X_i[:idx], X_s[:idx], Y[:idx]
    
    if len(X_i) == 0:
        raise ValueError(f"\n[ERROR] 0 file video dimuat dari '{path_to_use}'!")
        
    np.savez(PROCESSED_DATA_FILE, X_img=X_i, X_spec=X_s, y=Y)
    return X_i, X_s, Y

# ===============================
# BIMODAL FUSION LAYER (VISUAL + AUDIO)
# ===============================
@tf.keras.utils.register_keras_serializable()
class BimodalFusion(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.attention_net = models.Sequential([
            layers.Dense(32, activation='tanh'),
            layers.Dense(2, activation='softmax') # 2 bobot untuk Visual dan Audio
        ])
    def call(self, inputs):
        v, a = inputs
        combined = layers.concatenate([v, a])
        weights = self.attention_net(combined)
        wv, wa = weights[:, 0:1], weights[:, 1:2]
        fused_output = layers.concatenate([v * wv, a * wa])
        return [fused_output, weights]

# ===============================
# MODEL M6 (BIMODAL: AUDIO + VISUAL)
# ===============================
def build_bimodal_model():
    # Jalur Visual
    in_img = layers.Input(shape=(*IMG_SIZE, 3))
    base_cnn = EfficientNetB0(include_top=False, weights='imagenet', input_tensor=in_img)
    base_cnn.trainable = False
    v = layers.GlobalAveragePooling2D()(base_cnn.output)
    v = layers.BatchNormalization()(v)
    v = layers.Dense(128, activation='relu')(v)

    # Jalur Audio
    in_spec = layers.Input(shape=(*SPEC_SIZE, 1))
    a = layers.Conv2D(32, 3, padding='same', activation='relu')(in_spec)
    a = layers.BatchNormalization()(a)
    a = layers.MaxPooling2D()(a)
    a = layers.GlobalAveragePooling2D()(a)
    a = layers.Dense(128, activation='relu')(a)

    # Fusion
    fused, att_weights = BimodalFusion(name='fusion_layer')([v, a])
    
    x = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.02))(fused)
    x = layers.Dropout(0.5)(x)
    x = layers.BatchNormalization()(x)
    out = layers.Dense(1, activation='sigmoid', name='prediction')(x)

    model = models.Model([in_img, in_spec], out)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=2e-4), 
                  loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.1), 
                  metrics=['accuracy'])
    return model

# ===============================
# MAIN + CLASS WEIGHT + 5-FOLD CV
# ===============================
def main():
    X_i, X_s, Y = load_dataset_bimodal()

    class_weights = class_weight.compute_class_weight('balanced', classes=np.unique(Y), y=Y)
    class_weights_dict = {i: cw for i, cw in enumerate(class_weights)}
    print(f"Class weights: {class_weights_dict}")

    skf = StratifiedKFold(n_splits=KFOLD, shuffle=True, random_state=42)
    fold_histories = []
    
    final_model = None

    print("\nTraining Model M6 (Bimodal Audio + Visual)...")
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_i, Y)):
        print(f"\n{'='*40}")
        print(f"=== Fold {fold+1}/{KFOLD} ===")
        print(f"{'='*40}")
        
        X_i_tr, X_i_val = X_i[train_idx], X_i[val_idx]
        X_s_tr, X_s_val = X_s[train_idx], X_s[val_idx]
        Y_tr, Y_val = Y[train_idx], Y[val_idx]

        model = build_bimodal_model()
        history = model.fit(
            [X_i_tr, X_s_tr], Y_tr,
            validation_data=([X_i_val, X_s_val], Y_val),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            class_weight=class_weights_dict,
            verbose=1
        )
        fold_histories.append(history)

        # Plot akurasi & loss per fold
        plt.figure(figsize=(12,4))
        plt.subplot(1,2,1)
        plt.plot(history.history['accuracy'], label='train')
        plt.plot(history.history['val_accuracy'], label='val')
        plt.title(f'M6 (Audio+Visual) - Fold {fold+1} Acc')
        plt.legend()
        
        plt.subplot(1,2,2)
        plt.plot(history.history['loss'], label='train')
        plt.plot(history.history['val_loss'], label='val')
        plt.title(f'M6 (Audio+Visual) - Fold {fold+1} Loss')
        plt.legend()
        plt.show()

        # Evaluasi akhir tiap fold
        preds = model.predict([X_i_val, X_s_val])
        print(classification_report(Y_val, (preds>0.5).astype(int), target_names=['Non-Kasar', 'Kasar']))
        
        cm = confusion_matrix(Y_val, (preds>0.5).astype(int))
        plt.figure(figsize=(5,4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='OrRd', xticklabels=['Non-Kasar','Kasar'], yticklabels=['Non-Kasar','Kasar'])
        plt.title(f'Confusion Matrix M6 - Fold {fold+1}')
        plt.show()
        
        # Simpan model
        if fold == KFOLD - 1:
            final_model = model
            model.save(MODEL_FILE)
            
        if fold < KFOLD - 1:
            del model
            K.clear_session()
            gc.collect()

    # ==========================================
    # TESTING INFERENCE (AUDIO + VISUAL)
    # ==========================================
    fusion_layer_output = final_model.get_layer('fusion_layer').output[1] 
    inference_model = models.Model(inputs=final_model.input, outputs=[final_model.output, fusion_layer_output])

    print("\n\n>>> TESTING 10 FILES PER SUB-FOLDER (M6: AUDIO + VISUAL)")
    path_to_use = BASE_DATASET if os.path.exists(BASE_DATASET) else os.path.join(os.getcwd(), "Dataset")
    for main_cat in ['kasar', 'nonkasar']:
        cat_path = find_category_dir(path_to_use, main_cat)
        if not cat_path: continue
        
        subfolders = [d for d in os.listdir(cat_path) if os.path.isdir(os.path.join(cat_path, d))]
        if not subfolders: subfolders = [""]
        
        for sub in subfolders:
            sub_full = os.path.join(cat_path, sub) if sub else cat_path
            files = [f for f in os.listdir(sub_full) if f.lower().endswith((".mp4", ".avi", ".mkv"))][:10]
            if not files: continue
            
            print("="*80)
            print(f"Folder:  {sub if sub else main_cat}")
            print("="*80)
            
            for i, fname in enumerate(files, 1):
                img_a = extract_frame(os.path.join(sub_full, fname))
                spec_a = extract_audio_spectrogram(os.path.join(sub_full, fname))
                
                # Prediksi Bimodal
                pred, weights = inference_model.predict([
                    np.expand_dims(img_a,0), 
                    np.expand_dims(spec_a,0)
                ], verbose=0)
                
                p = pred[0][0]
                w_v, w_a = weights[0][0], weights[0][1]
                status = 'Kasar    ' if p >= 0.5 else 'Non-Kasar'
                
                print(f"{i:2}. [{status}]  {p*100:6.2f}%   | {fname[:35]}")
                print(f"    ↳ Weight ➔ Audio: {w_a:.2f} | Visual: {w_v:.2f}")
                
                del img_a, spec_a
                gc.collect()

if __name__ == "__main__":
    main()